**Github** : https://github.com/OzgurYldrm/AI-ML-Course     
**Youtube** : https://www.youtube.com/@F%C3%BCt%C3%BCrist_AIntelligence

https://tr.wikipedia.org/wiki/Vikipedi:Veritaban%C4%B1_indirme

In [1]:
import pandas as pd
import re
from collections import Counter
import torch
import numpy as np
import random
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", None)

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
device

device(type='cuda')

# Data

In [4]:
df = pd.read_csv("trwiki_random_2000.csv")

In [5]:
df

title  \
0                                                         Sedum patrickii   
1                                                      Caselle in Pittari   
2                                                           Dekan Yaylası   
3                                                             Pat O'Brien   
4                                             Musée de l'Aventure Peugeot   
...                                                                   ...   
1995  Vikipedi:Seçkin resim adayları/Plagiomnium affine laminazellen.jpeg   
1996                                 Karanlıkta Bir Çığlık (anlam ayrımı)   
1997                                          Kategori:Gambiya'daki çevre   
1998                               Kategori:Original Memphis Five üyeleri   
1999         Modül:Konum haritası/veri/ABD New Jersey Hudson County/belge   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

# Wiki Markup Temizleme

In [6]:
def wiki_clean(text):
    if pd.isna(text):
        return ""
    text = re.sub(r"#YÖNLENDİRME.*", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"<.*?>", " ", text, flags=re.DOTALL)
    text = re.sub(r"<ref.*?>.*?</ref>", " ", text, flags=re.DOTALL)
    text = re.sub(r"\{\{.*?\}\}", " ", text, flags=re.DOTALL)
    text = re.sub(r"\{\|.*?\|\}", " ", text, flags=re.DOTALL)
    text = re.sub(r"\[\[File:.*?\]\]", " ", text)
    text = re.sub(r"\[\[Dosya:.*?\]\]", " ", text)
    text = re.sub(r"\[\[.*?\|(.*?)\]\]", r"\1", text)
    text = re.sub(r"\[\[.*?\]\]", " ", text)
    text = text.replace("'''", " ")
    text = text.replace("''", " ")
    text = re.sub(r"\|.*?=", " ", text)
    text = re.sub(r"==.*?==", " ", text)
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"[^a-zA-ZçğıöşüÇĞİÖŞÜ\s]", " ", text)
    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()
    
    return text


In [7]:
df["text"] = df["text"].apply(wiki_clean)

In [8]:
df

,title,text
0,Sedum patrickii,sedum patrickii cinsine bağlı bir bitki türüdür
1,Caselle in Pittari,ülke caselle in pittari şehrinin raptiye harita boyutu campania i̇talya px caselle in pittari
2,Dekan Yaylası,
3,Pat O'Brien,müzisyen şarkı sözü yazarı gitar plak şirketi ın albümündeki render my prey adlı şarkının solo gitarında yer aldı de in solo çıkış albümü da yer alan race against disaster da gitardaydı aynı yıl in albümünde solo konuk olarak göründü tutuklanması aralık de ateşli silahla haneye tecavüz ve yaralama nedeniyle hapis cezasına ve ayrıca beş yıl gözetimin yanı sıra dolar para cezasına çarptırıldı mart ta o brien hapis cezasına çarptırıldı ve beş yıl gözetim altında tutulma cezası aldı ayrıca evinde yapılan araştırmada den fazla ateşli silah ve üç kafatası bulundu cannibal corpse ağ sayfası myspace te pat o brien
4,Musée de l'Aventure Peugeot,mus e de l aventure peugeot peugeot ailesi tarafından yılında müze
...,...,...
1995,Vikipedi:Seçkin resim adayları/Plagiomnium affine laminazellen.jpeg,mach iavelli msj mart utc thumb px
1996,Karanlıkta Bir Çığlık (anlam ayrımı),blake edwards ın yönettiği yapımı komedi filmi özgün adı a shot in the dark fred schepisi nin yönettiği yapımı dramatik film özgün adı a cry in the dark olan filmin diğer bir adı da evil angels dır
1997,Kategori:Gambiya'daki çevre,çevre
1998,Kategori:Original Memphis Five üyeleri,


# Vocab

In [9]:
df["tokens"] = df["text"].str.split()

In [15]:
all_tokens = [w for doc in df["tokens"] for w in doc]
counter = Counter(all_tokens)

min_count = 5
vocab = {w: c for w, c in counter.items() if c >= min_count}

word2idx = {w: i for i, w in enumerate(vocab.keys())}
idx2word = {i: w for w, i in word2idx.items()}

vocab_size = len(word2idx)
print("Vocab size:", vocab_size)

window_size = 5
n_neg = 5           
max_samples = 200000 # RAM koruma

pairs = []

for doc in df["tokens"]:

    indexed = [word2idx[w] for w in doc if w in word2idx]
    
    for i, center in enumerate(indexed):
        
        start = max(0, i - window_size)
        end = min(len(indexed), i + window_size + 1)
        
        context_words = indexed[start:i] + indexed[i+1:end]
        
        for context in context_words:
            
            pairs.append((center, context, 1))
    
            neg_count = 0
            while neg_count < n_neg:
                
                neg = random.randint(0, vocab_size - 1)
                if neg != context and neg != center:
                    pairs.append((center, neg, 0))
                    neg_count += 1
            
            if len(pairs) >= max_samples:
                break
        
        if len(pairs) >= max_samples:
            break
    
    if len(pairs) >= max_samples:
        break

binary_df = pd.DataFrame(
    pairs,
    columns=["center", "context", "label"]
)

print(binary_df.head())
print("Toplam örnek:", len(binary_df))
print("Positive oranı:", binary_df["label"].mean())


Vocab size: 2202
   center  context  label
0       0        1      1
1       0     1937      0
2       0     1098      0
3       0     1146      0
4       0      130      0
Toplam örnek: 200004
Positive oranı: 0.16666666666666666


# Dataset

In [17]:
class SkipGramDataset(Dataset):
    def __init__(self, dataframe):
        
        self.centers = torch.tensor(
            dataframe["center"].values,
            dtype=torch.long
        )
        
        self.contexts = torch.tensor(
            dataframe["context"].values,
            dtype=torch.long
        )
        
        self.labels = torch.tensor(
            dataframe["label"].values,
            dtype=torch.float32
        )
    
    def __len__(self):
        return len(self.centers)
    
    def __getitem__(self, idx):
        return (
            self.centers[idx],
            self.contexts[idx],
            self.labels[idx]
        )

In [19]:
batch_size = 1024
dataset = SkipGramDataset(binary_df)
loader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

# Model

In [20]:
class SkipGramBinary(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.input_embeddings = nn.Embedding(vocab_size, embedding_dim) # center embedding
        self.output_embeddings = nn.Embedding(vocab_size, embedding_dim) # context embedding
        self._init_embeddings()
    
    
    def _init_embeddings(self): # Word2Vec paper initialization
        initrange = 0.5 / self.input_embeddings.embedding_dim
        self.input_embeddings.weight.data.uniform_(-initrange, initrange)
        self.output_embeddings.weight.data.uniform_(-0, 0)
    
    def forward(self, center, context):
        v = self.input_embeddings(center)     # (B, D)
        u = self.output_embeddings(context)   # (B, D)
        score = torch.sum(v * u, dim=1)       # (B,)
        return score

In [22]:
vocab_size = len(word2idx)
embedding_dim = 300

model = SkipGramBinary(vocab_size, embedding_dim)
model.to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Training

In [23]:
for epoch in range(10):
    total_loss = 0
    for center, context, label in loader:
        center = center.to(device)
        context = context.to(device)
        label = label.to(device)
        optimizer.zero_grad()
        logits = model(center, context)
        loss = criterion(logits, label)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1} Loss: {total_loss/len(loader):.4f}")


Epoch 1 Loss: 0.4715
Epoch 2 Loss: 0.2638
Epoch 3 Loss: 0.2429
Epoch 4 Loss: 0.2272
Epoch 5 Loss: 0.2117
Epoch 6 Loss: 0.1949
Epoch 7 Loss: 0.1773
Epoch 8 Loss: 0.1593
Epoch 9 Loss: 0.1422
Epoch 10 Loss: 0.1266


# Evaluation

In [24]:
def get_embedding(word, model, word2idx, device="cpu"):
    if word not in word2idx:
        raise ValueError(f"{word} vocab içinde yok")
    
    model.eval()
    idx = torch.tensor(
        [word2idx[word]],
        dtype=torch.long,
        device=device
    )
    with torch.no_grad():
        embedding = model.input_embeddings(idx)
    
    return embedding.squeeze(0).cpu()

In [ ]:
vec = get_embedding("ankara", model, word2idx, device)
print(vec.shape)
print(vec)

In [38]:
word2idx

{'cinsine': 0,
 'bağlı': 1,
 'bir': 2,
 'bitki': 3,
 'türüdür': 4,
 'ülke': 5,
 'in': 6,
 'şehrinin': 7,
 'raptiye': 8,
 'harita': 9,
 'boyutu': 10,
 'i̇talya': 11,
 'px': 12,
 'müzisyen': 13,
 'şarkı': 14,
 'yazarı': 15,
 'gitar': 16,
 'şirketi': 17,
 'ın': 18,
 'my': 19,
 'adlı': 20,
 'şarkının': 21,
 'solo': 22,
 'yer': 23,
 'aldı': 24,
 'de': 25,
 'çıkış': 26,
 'albümü': 27,
 'da': 28,
 'alan': 29,
 'aynı': 30,
 'yıl': 31,
 'olarak': 32,
 'aralık': 33,
 've': 34,
 'nedeniyle': 35,
 'ayrıca': 36,
 'beş': 37,
 'yanı': 38,
 'sıra': 39,
 'dolar': 40,
 'para': 41,
 'mart': 42,
 'ta': 43,
 'o': 44,
 'altında': 45,
 'yapılan': 46,
 'den': 47,
 'fazla': 48,
 'üç': 49,
 'bulundu': 50,
 'sayfası': 51,
 'te': 52,
 'e': 53,
 'l': 54,
 'ailesi': 55,
 'tarafından': 56,
 'yılında': 57,
 'ile': 58,
 'olabilir': 59,
 'türkiye': 60,
 'ili': 61,
 'ilçesi': 62,
 'i̇zmir': 63,
 'ilçesine': 64,
 'mahalle': 65,
 'merkez': 66,
 'k': 67,
 'köy': 68,
 'belde': 69,
 'ankara': 70,
 'kütahya': 71,
 'bursa': 72

In [29]:
vec1 = get_embedding("ankara", model, word2idx, device)
vec2 = get_embedding("istanbul", model, word2idx, device)
vec3 = get_embedding("bitki", model, word2idx, device)

In [31]:
print(np.dot(vec1,vec2))
print(np.dot(vec1,vec3))

-0.0047788993
12.14783


/tmp/ipykernel_5541/22616535.py:1: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  print(np.dot(vec1,vec2))
/tmp/ipykernel_5541/22616535.py:2: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  print(np.dot(vec1,vec3))
